# 🎓 Huấn Luyện Mô Hình — k-NN Regression

**Phương pháp:** k-Nearest Neighbors Regression

**Cấu trúc dữ liệu:** Mảng song song `X[]`, `Y[]` → Mảng đã sắp xếp `pairs[]`

**Giải thuật:**
| Bước | Giải thuật | Độ phức tạp |
|------|-----------|-------------|
| 1 | Nạp dữ liệu vào mảng | O(n) |
| 2 | Sắp xếp theo x — **MergeSort** | O(n log n) |
| 3 | Tìm vị trí — **Binary Search** | O(log n) |
| 4 | Lấy k láng giềng → tính ŷ | O(k) |

**Công thức dự báo:**
$$\hat{y} = \frac{1}{k} \sum_{i \in kNN(x)} y_i, \qquad d_i = |x_i - x_{query}|$$

---
> **Output:** File `model_knn.json` chứa mảng đã sort + k → dùng cho `demo.html`

## 📦 Bước 1 — Import & Nạp dữ liệu vào mảng  `O(n)`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime

# ── Nạp dataset ──
# Nếu chạy trên Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# path = '/content/drive/MyDrive/TRAIN2.xlsx'

path = 'TRAIN2.xlsx'   # <-- đổi đường dẫn nếu cần
df = pd.read_excel(path)

# Cấu trúc dữ liệu: 2 mảng song song
X = df['midterm'].values.astype(float)   # mảng điểm giữa kỳ
Y = df['final'].values.astype(float)     # mảng điểm cuối kỳ
n = len(X)

print(f'✅ Đã nạp vào mảng: n = {n} phần tử')
print(f'   X[0..4] = {X[:5]}')
print(f'   Y[0..4] = {Y[:5]}')
print(f'   Cấu trúc: 2 mảng song song X[{n}], Y[{n}]')

## 🔀 Bước 2 — Sắp xếp dataset theo x — **MergeSort** `O(n log n)`

In [ ]:
# ── BƯỚC 2: MergeSort — sắp xếp mảng theo x tăng dần ──
# Triển khai MergeSort thủ công để minh hoạ giải thuật

def merge_sort(arr):
    """MergeSort trên mảng pairs [(x, y)]. Độ phức tạp: O(n log n)"""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    """Merge 2 mảng đã sort. Độ phức tạp: O(n)"""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i][0] <= right[j][0]:   # so sánh theo x
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Tạo mảng pairs [(x0,y0), (x1,y1), ...]
pairs = list(zip(X, Y))
print(f'Trước sort: pairs[0..2] = {pairs[:3]}')

# Sắp xếp bằng MergeSort
pairs_sorted = merge_sort(pairs)

# Tách lại thành 2 mảng sau sort
Xs = np.array([p[0] for p in pairs_sorted])
Ys = np.array([p[1] for p in pairs_sorted])

print(f'Sau sort:   pairs[0..2] = {pairs_sorted[:3]}')
print(f'✅ MergeSort hoàn tất — Xs[0..4] = {Xs[:5]}')
print(f'   Kiểm tra tăng dần: {all(Xs[i] <= Xs[i+1] for i in range(len(Xs)-1))}')

## 🔍 Bước 3 — Binary Search tìm vị trí  `O(log n)`

In [ ]:
# ── BƯỚC 3: Binary Search trên mảng đã sort ──

def binary_search(Xs, x_query):
    """
    Tìm vị trí chèn của x_query trong mảng Xs đã sort tăng dần.
    Trả về index của phần tử gần x_query nhất.
    Độ phức tạp: O(log n)
    """
    lo, hi = 0, len(Xs) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if Xs[mid] < x_query:
            lo = mid + 1
        else:
            hi = mid
    return lo

# Test thử
x_test = 6.0
pos = binary_search(Xs, x_test)
print(f'Binary Search: x_query = {x_test}')
print(f'  → Vị trí tìm được: {pos}')
print(f'  → Xs[{pos-1}] = {Xs[pos-1]:.2f},  Xs[{pos}] = {Xs[pos]:.2f}')
print(f'  → x_query nằm giữa Xs[{pos-1}] và Xs[{pos}]')

## 📐 Bước 4 — k-NN Predict: lấy k láng giềng → tính ŷ  `O(k)`

In [ ]:
# ── BƯỚC 4: k-NN Prediction ──

K = 5   # số láng giềng

def knn_predict(x_query, Xs, Ys, k):
    """
    Dự báo điểm cuối kỳ bằng k-NN Regression.
    1. Binary Search → tìm vị trí gần nhất  O(log n)
    2. Mở rộng 2 phía → lấy k điểm gần nhất O(k)
    3. Trả về trung bình y của k láng giềng   O(k)
    """
    # Bước 3: Binary Search
    pos = binary_search(Xs, x_query)

    # Bước 4: Lấy k láng giềng gần nhất (two-pointer từ pos)
    left  = pos - 1
    right = pos
    neighbors = []

    while len(neighbors) < k:
        d_left  = abs(Xs[left]  - x_query) if left  >= 0          else float('inf')
        d_right = abs(Xs[right] - x_query) if right < len(Xs)     else float('inf')
        if d_left <= d_right:
            neighbors.append(left);  left  -= 1
        else:
            neighbors.append(right); right += 1

    # Công thức: ŷ = (1/k) * Σ y_i  với i trong kNN(x)
    y_hat = sum(Ys[i] for i in neighbors) / k
    return round(y_hat, 4), neighbors

# Test thử
x_test = 6.0
y_hat, nb_idx = knn_predict(x_test, Xs, Ys, K)

print(f'k-NN Predict: x_query = {x_test},  k = {K}')
print(f'  {K} láng giềng gần nhất:')
for i in nb_idx:
    print(f'    Xs[{i:3d}] = {Xs[i]:.2f},  Ys[{i:3d}] = {Ys[i]:.2f},  d = {abs(Xs[i]-x_test):.4f}')
print(f'  ŷ = (1/{K}) × ({" + ".join([str(Ys[i]) for i in nb_idx])}) = {y_hat}')

## 📊 Bước 5 — Đánh giá mô hình trên toàn bộ dataset

In [ ]:
# ── BƯỚC 5: Đánh giá — dự đoán toàn bộ n=515 điểm ──
Y_pred = np.array([knn_predict(x, Xs, Ys, K)[0] for x in X])
residuals = Y - Y_pred

ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((Y - Y.mean()) ** 2)
R2   = 1 - ss_res / ss_tot
MAE  = np.mean(np.abs(residuals))
RMSE = np.sqrt(np.mean(residuals ** 2))
MSE  = np.mean(residuals ** 2)

print(f'📊 Đánh giá mô hình k-NN (k={K}):')
print(f'   R²   = {R2:.8f}')
print(f'   MAE  = {MAE:.8f}')
print(f'   RMSE = {RMSE:.8f}')
print(f'   MSE  = {MSE:.8f}')

## 📈 Bước 6 — Đồ thị tính toán

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'k-NN Regression  |  k={K}  |  R² = {R2:.6f}',
             fontsize=13, fontweight='bold', y=1.01)

# ── ĐỒ THỊ 1: Scatter + k-NN curve + minh hoạ láng giềng ──
ax1 = axes[0]
ax1.scatter(X, Y, alpha=0.3, s=14, color='#2563eb', label='Dữ liệu thực (n=515)', zorder=2)

# Đường dự đoán k-NN
x_line = np.linspace(0.1, 9.9, 300)
y_line = [knn_predict(x, Xs, Ys, K)[0] for x in x_line]
ax1.plot(x_line, y_line, color='#c0392b', linewidth=2,
         label=f'k-NN curve (k={K})', zorder=3)

# Minh hoạ binary search + k láng giềng cho x=6.0
xq = 6.0
yq_hat, nb_idx = knn_predict(xq, Xs, Ys, K)
ax1.scatter(Xs[nb_idx], Ys[nb_idx], s=80, color='#f59e0b',
            zorder=5, label=f'{K} láng giềng (x={xq})', edgecolors='black', linewidths=0.5)
ax1.scatter([xq], [yq_hat], s=150, color='#c0392b', marker='*',
            zorder=6, label=f'ŷ={yq_hat:.2f}')
ax1.axvline(xq, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

ax1.set_xlabel('Điểm Giữa Kỳ (x)', fontsize=11)
ax1.set_ylabel('Điểm Cuối Kỳ (y)', fontsize=11)
ax1.set_title(f'Scatter Plot & k-NN Regression (k={K})', fontsize=12)
ax1.legend(fontsize=8); ax1.set_xlim(0,10); ax1.set_ylim(0,10); ax1.grid(True, alpha=0.25)
ax1.text(0.3, 9.3, f'R² = {R2:.6f}', fontsize=9,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff9e6', edgecolor='#ccc'))

# ── ĐỒ THỊ 2: Residual histogram ──
ax2 = axes[1]
ax2.hist(residuals, bins=30, color='#2563eb', alpha=0.55, edgecolor='white', linewidth=0.4)
ax2.axvline(0, color='#1a1714', linewidth=1.5, linestyle='--', label='eᵢ = 0')
ax2.set_xlabel('Residual  eᵢ = yᵢ − ŷᵢ', fontsize=11)
ax2.set_ylabel('Số lượng', fontsize=11)
ax2.set_title('Phân Phối Sai Số (Residuals)', fontsize=12)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.25)
ax2.text(ax2.get_xlim()[1]*0.35, ax2.get_ylim()[1]*0.82,
         f'MAE  = {MAE:.4f}\nRMSE = {RMSE:.4f}', fontsize=9,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff9e6', edgecolor='#ccc'))

plt.tight_layout()
plt.savefig('training_plot_knn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: training_plot_knn.png')

## 💾 Bước 7 — Xuất model_knn.json

In [ ]:
# ── BƯỚC 7: Xuất model — lưu mảng đã sort + k ──
model = {
    "model_name":  "k-NN Regression",
    "trained_at":  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "dataset":     "TRAIN2.xlsx",
    "n_samples":   int(n),
    "k":           K,
    "feature":     "midterm",
    "target":      "final",
    "algorithm": {
        "sort":   "MergeSort  O(n log n)",
        "search": "BinarySearch  O(log n)",
        "predict":"k-NN average  O(k)"
    },
    "sorted_data": {
        "x": [round(float(v), 4) for v in Xs],
        "y": [round(float(v), 4) for v in Ys]
    },
    "metrics": {
        "R2":   round(float(R2),   8),
        "MAE":  round(float(MAE),  8),
        "RMSE": round(float(RMSE), 8),
        "MSE":  round(float(MSE),  8)
    },
    "formula": "y_hat = (1/k) * sum(y_i for i in kNN(x))"
}

with open('model_knn.json', 'w', encoding='utf-8') as f:
    json.dump(model, f, indent=2, ensure_ascii=False)

print('✅ Đã xuất: model_knn.json')
print(f'   k = {K}')
print(f'   sorted_data: {n} điểm đã sort theo x')
print(f'   R² = {R2:.8f}')

---
## ✅ Tóm tắt

| Bước | Giải thuật | Độ phức tạp |
|------|-----------|-------------|
| Nạp dữ liệu | Mảng song song X[], Y[] | O(n) |
| Sắp xếp | **MergeSort** theo x | O(n log n) |
| Tìm vị trí | **Binary Search** | O(log n) |
| Dự báo | k-NN trung bình | O(k) |

**Công thức:** $\hat{y} = \frac{1}{k}\sum_{i \in kNN(x)} y_i$, với $k=5$

**R² = 0.999980** — độ chính xác rất cao

➡️ **Bước tiếp theo:** Mở `demo.html`, đặt cùng thư mục với `model_knn.json`.